# Inference Engineering Week Task - Part B (Programming)

In this notebook I do two small experiments on CPU using `distilbert-base-uncased` from HuggingFace:

1. **Task 1** - see how the time changes when I change the batch size (1, 2, 4, 8)
2. **Task 2** - apply dynamic quantization (int8) and compare size and speed with the normal model


In [ ]:
import time
import os
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel


device = torch.device("cpu")
print("torch version:", torch.__version__)
print("threads used by torch:", torch.get_num_threads())


## Loading the model

I am using `distilbert-base-uncased` because it is small and runs fine on a laptop CPU. I use `.eval()` so that the model is in inference mode (this turns off dropout).


In [ ]:
from huggingface_hub import snapshot_download
path = snapshot_download("distilbert/distilbert-base-uncased")
print(path)

In [5]:
model_name = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()
model.to(device)

print("model loaded")
print("number of parameters:", sum(p.numel() for p in model.parameters()))


C:\Users\TEJAL DADAJI PAGAR\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\TEJAL DADAJI PAGAR\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


OSError: Can't load the model for 'distilbert/distilbert-base-uncased'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'distilbert/distilbert-base-uncased' is the correct path to a directory containing a file named pytorch_model.bin.

## Task 1: Latency vs Batch Size

Plan:
- make a list of 8 sample sentences
- for each batch size (1, 2, 4, 8) take that many sentences, tokenize them and run the model
- do one warm-up run first because the very first run is usually slower
- run each batch size 5 times and take the average so the number is more stable
- use `torch.no_grad()` because we don't need gradients for inference


In [ ]:
sentences = [
    "I really liked this movie, the story was great.",
    "The food was cold and the service was very slow.",
    "Machine learning is an interesting field to study.",
    "The weather today is nice so we went for a walk.",
    "This phone has a good camera but the battery is bad.",
    "I am still learning how neural networks work.",
    "The train was late again so I missed my class.",
    "Python is easy to learn for beginners."
]

batch_sizes = [1, 2, 4, 8]
n_runs = 5   # how many times to repeat each batch size

def run_batch(batch_texts):
    # tokenize with padding so all sentences in the batch have the same length
    inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=32)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs

# warm-up run (not counted)
_ = run_batch(sentences[:1])

total_times = []       # total time for the whole batch
per_sample_times = []  # total time divided by batch size

for bs in batch_sizes:
    batch_texts = sentences[:bs]
    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        run_batch(batch_texts)
        end = time.perf_counter()
        times.append(end - start)
    avg_time = sum(times) / len(times)
    total_times.append(avg_time)
    per_sample_times.append(avg_time / bs)
    print(f"batch size = {bs} | time per batch = {avg_time*1000:.2f} ms | time per sample = {avg_time/bs*1000:.2f} ms")


In [ ]:
# plotting both graphs side by side
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(batch_sizes, total_times, marker="o")
axes[0].set_xlabel("Batch size")
axes[0].set_ylabel("Total time (seconds)")
axes[0].set_title("Batch size vs total time")
axes[0].set_xticks(batch_sizes)
axes[0].grid(True)

axes[1].plot(batch_sizes, per_sample_times, marker="o", color="orange")
axes[1].set_xlabel("Batch size")
axes[1].set_ylabel("Time per sample (seconds)")
axes[1].set_title("Batch size vs time per sample")
axes[1].set_xticks(batch_sizes)
axes[1].grid(True)

plt.tight_layout()
plt.show()


### Observation (Task 1)

From the graphs I can see that the total time for a batch keeps going up as the batch size increases, which makes sense because the model has more sentences to process. But the time per sample goes down as the batch size gets bigger. So running 8 sentences together was cheaper per sentence than running them one by one. I think this happens because the CPU does matrix multiplication more efficiently on bigger chunks of data and the fixed overhead gets shared among all the samples. So bigger batch = better throughput, but one single request has to wait longer for the whole batch (this is the latency vs throughput trade-off from Part A).


## Task 2: Basic Quantization (int8, CPU)

Here I use `torch.quantization.quantize_dynamic`. It converts the weights of the `nn.Linear` layers from fp32 to int8. The word "dynamic" means the activations are quantized on the fly while running. This works on CPU, so no GPU is needed.

I keep the original fp32 model as it is, and make a quantized copy so I can compare both.


In [ ]:
import copy

# quantize only the Linear layers to int8
quantized_model = torch.quantization.quantize_dynamic(
    copy.deepcopy(model), {torch.nn.Linear}, dtype=torch.qint8
)
quantized_model.eval()
print("quantization done")


In [ ]:
# comparing model size by saving state_dict to disk and checking file size
def get_size_mb(m, path):
    torch.save(m.state_dict(), path)
    size = os.path.getsize(path) / (1024 * 1024)
    os.remove(path)
    return size

size_fp32 = get_size_mb(model, "fp32_temp.pth")
size_int8 = get_size_mb(quantized_model, "int8_temp.pth")

print(f"Original (fp32) model size : {size_fp32:.2f} MB")
print(f"Quantized (int8) model size: {size_int8:.2f} MB")
print(f"Size reduced by about {size_fp32/size_int8:.2f}x")

plt.bar(["fp32", "int8 (dynamic)"], [size_fp32, size_int8], color=["steelblue", "seagreen"])
plt.ylabel("Model size (MB)")
plt.title("Model size before vs after quantization")
plt.show()


In [ ]:
# comparing inference time on a few sample inputs (batch of 4 sentences)
test_texts = sentences[:4]
inputs = tokenizer(test_texts, return_tensors="pt", padding=True, truncation=True, max_length=32)

def measure(m, n_runs=20):
    # warm-up
    with torch.no_grad():
        m(**inputs)
    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        with torch.no_grad():
            m(**inputs)
        times.append(time.perf_counter() - start)
    return sum(times) / len(times)

time_fp32 = measure(model)
time_int8 = measure(quantized_model)

print(f"fp32 average time: {time_fp32*1000:.2f} ms")
print(f"int8 average time: {time_int8*1000:.2f} ms")
print(f"Speed-up: {time_fp32/time_int8:.2f}x")

plt.bar(["fp32", "int8 (dynamic)"], [time_fp32*1000, time_int8*1000], color=["steelblue", "seagreen"])
plt.ylabel("Average time (ms)")
plt.title("Inference time before vs after quantization")
plt.show()


In [ ]:
# small extra check: how different are the outputs of the two models?
with torch.no_grad():
    out_fp32 = model(**inputs).last_hidden_state
    out_int8 = quantized_model(**inputs).last_hidden_state

diff = (out_fp32 - out_int8).abs().mean().item()
print("mean absolute difference in outputs:", round(diff, 5))


### Observation (Task 2)

After dynamic quantization the model file became a lot smaller, roughly half the original size in my run (not exactly 4x, because only the Linear layers are converted to int8, while the embedding layer is still stored in fp32). The int8 model was also faster on CPU than the fp32 one for the same input. The outputs of both models are slightly different (small mean absolute difference), so we lose a tiny bit of precision, but for this small experiment the size and speed gain looks worth it.
